In [1]:
import trajectory_data
from dataloader import TrajectoryDataset
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dino_model import DINOStateDecider
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from trajectory_data import skill_visualizer
from gesture_detector.utils import pretty_confusion_matrix
import torch

from video_embedding.image_processing import saved_img_processing

def predict(decider, X_train, y_train):
    y_true, y_pred = [], []
    for x,y in zip(X_train, y_train):
        is_known, y_p = decider.predict(x) #, timestep=0.0)
        # -> (True, branch_id) or (False, -1)
        y_true.append(y)
        y_pred.append(y_p)
    return y_true, y_pred

In [3]:
dataset = TrajectoryDataset(trajectory_data.package_path)

In [ ]:
# decider = DINOStateDecider(dino_variant="dinov2_vits14", use_cls_token=True, batch_size=128, percent_keep=0.05)
decider = StateDeciderSIFT()

# Door Open/Close classification

In [ ]:
X_train, Xt_train, y_train = dataset.get_image_dataset(dataset.get_all_names("user_0_kine_peg_pick"))

NameError: name 'dataset' is not defined

In [ ]:

y_train = torch.where(y_train < 10, torch.tensor(0), torch.tensor(1))
# X_test, y_test = dataset.get_image_dataset(video_test_names, slice(50,80))
# y_test = torch.where(y_test < 10, torch.tensor(0), torch.tensor(1))

In [ ]:
dataset.get_task_graph_structure('user_0_kine_peg_pick')

In [ ]:
# plot task-graph
from nocode_robot_programming.state_decision.task_graph import get_task_graph_structure
get_task_graph_structure(lfd, 'user_0_kine_peg_pick')

In [ ]:
decider.train(X_train, y_train)  # X_train: (N,H,W) or (N,H,W,3), y_train: (N,)
y_true, y_pred = predict(decider, X_test, y_test)
pretty_confusion_matrix.pp_matrix_from_data(y_true, y_pred, figsize=(6,6), columns=["close", "open"])

In [ ]:
skill_visualizer.run()

In [ ]:
from video_embedding.utils import set_session, get_all_names
from video_embedding.models.video_embedder import VideoEmbedder
from video_embedding.models.video_embedding_dataset import load_dataloader

from risk_estimation.result_evaluator import ResultEvaluator
from risk_estimation.datasets.risk_feature_extractor import *
from risk_estimation.datasets.risk_dataloader import RiskEstimationDataset as D
from risk_estimation.datasets.frame_dropping import *
from risk_estimation.models.mlp_risk_estimator import MLPRiskEstimator, MLPRiskEstimator2
from risk_estimation.models.gp_risk_estimator import GPRiskEstimator, TwinGPRiskEstimator
from risk_estimation.models.dist_risk_estimator import *
from risk_estimation.models.resnet_risk_estimator import ResNetRiskEstimator
from risk_estimation.models.safety_layer import get_risk_estimator
from video_embedding.utils import all_trial_names, all_test_names, visualize_labelled_video, visualize_labelled_video_frame, get_session
from video_embedding.models.video_embedder import VideoEmbedder

from risk_estimation.models.risk_estimator import *
from video_embedding.models.nerual_networks.autoencoder import *

In [ ]:

class StateDeciderAEGP:
    def __init__(self, name_skill: str = "model_test_name_123456"):


    def train(self, X, y):
        pass

    def predict(self, img):
        x, _ = StampedLatentObservationsRiskLabels.extract({'img': saved_img_processing(X_train[0:1]), "frame_number": torch.tensor([[0.0]]), "risk_flag": torch.tensor([[0.0]])}, video_embedder)
        system_risk_pred, risk, std = self.risk_estimator.sample(torch.tensor(x).cuda())
        system_risk_pred = int(np.array(system_risk_pred).squeeze())
        risk = float(np.array(risk).squeeze())
        
        return system_risk_pred

In [ ]:
class StateDeciderAEGP:
    pass


In [ ]:
self = StateDeciderAEGP()


In [ ]:

self.video_embedder = VideoEmbedder(
    name=name_skill,
    latent_dim=12,
    learning_rate=0.001,
    nn_model="Autoencoder3",
)
videoembed_dataloader = load_dataloader(video_train_names, batch_size=64)


In [ ]:

self.video_embedder.train(videoembed_dataloader, video_train_names, num_epochs=50)
self.video_embedder.save_model()

dataloader = D.load_dataloader(video_train_names, self.video_embedder, 40, OnlyLabelledFramesDroppingPolicy, StampedLatentObservationsRiskLabels)



self.risk_estimator = get_risk_estimator(
    approach="GP",
    skill_name=self.video_embedder.name,
    xdim=StampedLatentObservationsRiskLabels.xdim(12),
    video_embedder=self.video_embedder,
    out_assessment="cautious",
    train_epoch = 400,
)
self.risk_estimator.set_dataloaders_for_validation([
        dataloader,
    ], names=["test"])
self.risk_estimator.training_loop(dataloader)
